# 02_baseline_cnn.ipynb — HistoBreastNet (Persona 1)

CNN **addestrata da zero** come baseline per la classificazione binaria benigno/maligno.
Il notebook **consuma** gli split e le fold prodotti da `01_preprocessing.ipynb`
(config `diversity_1p5GB`) e **non li rigenera**. Persona 2 userà gli stessi file.

**Cosa produce**
- Baseline CNN valutata su tre protocolli: image-wise, patient-wise, k-fold patient-wise (k=5).
- Metriche predittive: accuracy, precision (macro), recall (macro), F1 (macro), AUROC,
  recall della classe maligna (sensibilita clinica), confusion matrix.
- Metriche di efficienza per D1: n. parametri, tempo totale/medio-per-epoca di training,
  tempo di inferenza per immagine, dimensione del modello su disco.
- Tabella finale `results/tables/cnn_baseline_metrics.csv` (schema fisso) + sintesi k-fold (media +/- std).

**Nota D2.** Con pochi pazienti il singolo split patient-wise ha varianza alta: il riferimento
per la generalizzazione su pazienti non visti e la **k-fold patient-wise** (media +/- std), non il singolo split.

## Sezione 0 — Setup, path, riproducibilità

In [ ]:
import os, sys, gc, time, json, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from google.colab import drive
drive.mount('/content/drive')

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models

# ----- Riproducibilita ------------------------------------------------------
RANDOM_STATE = 42
os.environ['PYTHONHASHSEED'] = str(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
tf.random.set_seed(RANDOM_STATE)
keras.utils.set_random_seed(RANDOM_STATE)
# Determinismo completo (piu lento): scommenta se serve bit-per-bit reproducibility.
# tf.config.experimental.enable_op_determinism()

# ----- Path progetto --------------------------------------------------------
PROJECT_ROOT   = Path('/content/drive/MyDrive/HistoBreastNet')
DATA_PROCESSED = PROJECT_ROOT/'data'/'processed'
CONFIG_NAME    = 'diversity_1p5GB'            # config principale per la baseline
CONFIG_DIR     = DATA_PROCESSED/CONFIG_NAME
RESULTS_DIR     = PROJECT_ROOT/'results'
TABLES_DIR      = RESULTS_DIR/'tables'
FIGURES_DIR     = RESULTS_DIR/'figures'
EXPERIMENTS_DIR = PROJECT_ROOT/'experiments'   # una sottocartella per run (linee guida Sez. 4.2.4)
for d in (TABLES_DIR, FIGURES_DIR, EXPERIMENTS_DIR):
    d.mkdir(parents=True, exist_ok=True)

assert CONFIG_DIR.exists(), f'Config non trovata: {CONFIG_DIR} (esegui prima 01_preprocessing).'

# ----- Iperparametri baseline ----------------------------------------------
IMG_SIZE   = 224
BATCH_SIZE = 32
EPOCHS     = 40            # limite superiore; EarlyStopping ferma prima
PATIENCE   = 8
QUICK_TEST = False         # True = giro rapido per validare la pipeline (2 epoche)
if QUICK_TEST:
    EPOCHS, PATIENCE = 2, 2

AUTOTUNE = tf.data.AUTOTUNE
gpus = tf.config.list_physical_devices('GPU')
print('TF', tf.__version__, '| GPU:', gpus if gpus else 'NESSUNA (attiva runtime GPU su Colab)')
print('Config baseline:', CONFIG_NAME)

## Sezione 1 — Disponibilita del dataset in locale

Gli split contengono `relative_path` rispetto alla root del dataset. Per addestrare servono
i PNG in locale: si riusa lo stesso ZIP su Drive creato dal `01`. Se il runtime si disconnette,
rilancia questa cella (l'estrazione riparte da capo, 1-3 min).

In [ ]:
import zipfile, shutil

BREAKHIS_LOCAL = Path('/content/breakhis/BreaKHis_v1')                  # root dei relative_path
BREAKHIS_ZIP   = PROJECT_ROOT/'data'/'original'/'BreaKHis_v1.zip'

def n_local():
    return sum(1 for _ in BREAKHIS_LOCAL.rglob('*.png')) if BREAKHIS_LOCAL.exists() else 0

n = n_local()
if n == 7909:
    print('Dataset gia completo in locale:', n)
elif BREAKHIS_ZIP.exists():
    shutil.rmtree('/content/breakhis', ignore_errors=True)
    BREAKHIS_LOCAL.mkdir(parents=True, exist_ok=True)
    t0 = time.time(); print('Estrazione dallo zip su Drive...')
    with zipfile.ZipFile(BREAKHIS_ZIP) as z:
        z.extractall(BREAKHIS_LOCAL)
    n = n_local(); print('Estratto in %.0fs -> %d immagini' % (time.time()-t0, n))
else:
    raise FileNotFoundError('Ne dataset locale ne zip su Drive: esegui prima 01_preprocessing.')

assert n == 7909, 'Conteggio inatteso: %d (atteso 7909).' % n

## Sezione 2 — Caricamento split/fold e ricostruzione dei path

Si leggono i CSV della config e si ricostruisce il path assoluto da `relative_path`
(`full_path = BREAKHIS_LOCAL / relative_path`). I path assoluti del `01` (colonna `path`)
vengono ignorati: cosi il notebook e portabile tra sessioni.

In [ ]:
def add_full_path(df):
    """Aggiunge full_path da relative_path e resetta l'indice (ordine stabile)."""
    df = df.copy()
    df['full_path'] = df['relative_path'].map(lambda r: str(BREAKHIS_LOCAL / r))
    return df.reset_index(drop=True)

# Sanity: i file referenziati esistono davvero
_probe = add_full_path(pd.read_csv(CONFIG_DIR/'metadata_subset.csv'))
_missing = [p for p in _probe['full_path'].sample(min(50, len(_probe)), random_state=RANDOM_STATE)
            if not Path(p).exists()]
assert not _missing, f'File non trovati (esempio): {_missing[:3]}'
print('metadata_subset:', len(_probe), 'immagini |', _probe['patient_id'].nunique(), 'pazienti')
print('Distribuzione classi (immagini):', _probe['binary_label'].value_counts().to_dict(),
      ' (0=benign, 1=malignant)')

with open(CONFIG_DIR/'config.json') as f:
    CFG = json.load(f)
print('config.json ->', {k: CFG[k] for k in
      ('n_patients','n_images','actual_size_gib','n_patients_benign','n_patients_malignant')})

## Sezione 3 — Pipeline `tf.data`

Decodifica PNG -> resize 224x224 -> cache (uint8, memoria-efficiente) -> normalizzazione [0,1].
L'augmentation (flip/brightness/contrast leggeri) e applicata **solo al training**.
Val e test non sono mescolati: l'ordine delle predizioni combacia con quello dei DataFrame.

In [ ]:
def _decode_uint8(path, label):
    img = tf.io.read_file(path)
    img = tf.io.decode_png(img, channels=3)
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE], method='bilinear')
    img = tf.cast(img, tf.uint8)                      # cache leggera (~1 byte/px)
    return img, label

def _to_float(img, label):
    return tf.cast(img, tf.float32) / 255.0, label

def _augment(img, label):
    img = tf.image.random_flip_left_right(img)
    img = tf.image.random_flip_up_down(img)
    img = tf.image.random_brightness(img, 0.05)
    img = tf.image.random_contrast(img, 0.95, 1.05)
    img = tf.clip_by_value(img, 0.0, 1.0)
    return img, label

def make_ds(df, training=False, augment=False, batch=BATCH_SIZE):
    paths  = df['full_path'].values
    labels = df['binary_label'].values.astype('float32')
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    ds = ds.map(_decode_uint8, num_parallel_calls=AUTOTUNE).cache()
    if training:
        ds = ds.shuffle(min(len(df), 2048), seed=RANDOM_STATE, reshuffle_each_iteration=True)
    ds = ds.map(_to_float, num_parallel_calls=AUTOTUNE)
    if augment:
        ds = ds.map(_augment, num_parallel_calls=AUTOTUNE)
    return ds.batch(batch).prefetch(AUTOTUNE)

## Sezione 4 — Architettura CNN da zero e training

CNN volutamente compatta (4 blocchi Conv-BN-ReLU-Pool + GAP + testa densa) adeguata a un
dataset piccolo, per limitare l'overfitting. Uscita sigmoide (binaria). `class_weight` bilanciato
per non penalizzare la classe maligna. EarlyStopping su `val_auc` con ripristino dei pesi migliori.

In [ ]:
from sklearn.utils.class_weight import compute_class_weight

def build_baseline_cnn(input_shape=(IMG_SIZE, IMG_SIZE, 3), seed=RANDOM_STATE):
    keras.utils.set_random_seed(seed)               # init pesi riproducibile
    inp = layers.Input(shape=input_shape)
    x = inp
    for f in (32, 64, 128, 128):
        x = layers.Conv2D(f, 3, padding='same', use_bias=False)(x)
        x = layers.BatchNormalization()(x)
        x = layers.Activation('relu')(x)
        x = layers.MaxPooling2D()(x)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.40)(x)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.30)(x)
    out = layers.Dense(1, activation='sigmoid')(x)
    return models.Model(inp, out, name='baseline_cnn')

class TimeHistory(keras.callbacks.Callback):
    """Registra il tempo di ogni epoca per calcolare avg_epoch_time_sec."""
    def on_train_begin(self, logs=None): self.times = []
    def on_epoch_begin(self, epoch, logs=None): self._t = time.time()
    def on_epoch_end(self, epoch, logs=None): self.times.append(time.time() - self._t)

def class_weights_from(df):
    y = df['binary_label'].values
    w = compute_class_weight('balanced', classes=np.array([0, 1]), y=y)
    return {0: float(w[0]), 1: float(w[1])}

def train_cnn(train_df, val_df, seed=RANDOM_STATE):
    train_ds = make_ds(train_df, training=True, augment=True)
    val_ds   = make_ds(val_df,   training=False, augment=False)
    model = build_baseline_cnn(seed=seed)
    model.compile(optimizer=keras.optimizers.Adam(1e-3),
                  loss='binary_crossentropy',
                  metrics=[keras.metrics.AUC(name='auc'), 'accuracy'])
    th = TimeHistory()
    cbs = [keras.callbacks.EarlyStopping(monitor='val_auc', mode='max',
                                         patience=PATIENCE, restore_best_weights=True),
           keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                                             patience=4, min_lr=1e-5),
           th]
    t0 = time.time()
    hist = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS,
                     class_weight=class_weights_from(train_df), callbacks=cbs, verbose=2)
    train_time = time.time() - t0
    return model, hist, th, train_time

## Sezione 5 — Metriche predittive, efficienza, valutazione

**Convenzione metriche.** `precision`, `recall`, `f1` sono **macro** (media sulle due classi,
vista bilanciata); `recall_malignant` e la recall della sola classe maligna (sensibilita,
metrica clinicamente critica). `auroc` sulle probabilita. Confusion matrix salvata come tn/fp/fn/tp.

**Efficienza (D1).** `n_params` dal modello; `training_time_sec` a muro; `avg_epoch_time_sec`
media reale delle epoche eseguite; `inference_time_ms_per_image` throughput batch a regime
(un warm-up prima della misura); `model_size_mb` dal file `.keras` salvato su disco.

In [ ]:
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix)
from datetime import datetime

# --- Esperimenti: una cartella per run in experiments/ (Sez. 4.2.4) ---------
# Nome cartella ispirato a Persona 2:
#   <YYYYMMDD_HHMMSS_micros>_<dataset_config>_<split_type>_baseline_cnn_scratch_fold<N>
# Contenuto: config.json, model.keras, training_log.csv, metrics.csv
def make_experiment_dir(split_type, fold):
    fold_tag = 'NA' if pd.isna(fold) else int(fold)
    ts = datetime.now().strftime('%Y%m%d_%H%M%S_%f')
    name = f'{ts}_{CONFIG_NAME}_{split_type}_baseline_cnn_scratch_fold{fold_tag}'
    d = EXPERIMENTS_DIR/name
    d.mkdir(parents=True, exist_ok=True)
    return d

def save_training_log(exp_dir, hist, th):
    log = pd.DataFrame(hist.history)
    log.insert(0, 'epoch', range(1, len(log) + 1))
    if th.times and len(th.times) == len(log):
        log['epoch_time_sec'] = th.times
    log.to_csv(exp_dir/'training_log.csv', index=False)

def save_experiment_config(exp_dir, split_type, fold, sizes):
    cfg = dict(
        dataset_config=CONFIG_NAME, split_type=split_type,
        fold=(None if pd.isna(fold) else int(fold)),
        model='baseline_cnn', training_mode='scratch',
        image_size=[IMG_SIZE, IMG_SIZE], batch_size=BATCH_SIZE, seed=RANDOM_STATE,
        max_epochs=EPOCHS, patience=PATIENCE,
        optimizer='adam', learning_rate=1e-3, loss='binary_crossentropy',
        class_weight='balanced', quick_test=QUICK_TEST,
        source_config_dir=str(CONFIG_DIR),
        **sizes,
    )
    with open(exp_dir/'config.json', 'w') as f:
        json.dump(cfg, f, indent=2)


def compute_metrics(y_true, y_prob, thr=0.5):
    y_pred = (y_prob >= thr).astype(int)
    out = dict(
        accuracy=accuracy_score(y_true, y_pred),
        precision=precision_score(y_true, y_pred, average='macro', zero_division=0),
        recall=recall_score(y_true, y_pred, average='macro', zero_division=0),
        f1=f1_score(y_true, y_pred, average='macro', zero_division=0),
        recall_malignant=recall_score(y_true, y_pred, pos_label=1, zero_division=0),
    )
    try:
        out['auroc'] = roc_auc_score(y_true, y_prob)
    except ValueError:                                # test con una sola classe
        out['auroc'] = np.nan
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    out.update(tn=int(tn), fp=int(fp), fn=int(fn), tp=int(tp))
    return out

def measure_inference_ms(model, test_ds, n_images):
    _ = model.predict(test_ds, verbose=0)             # warm-up (grafo/JIT)
    t0 = time.perf_counter()
    _ = model.predict(test_ds, verbose=0)
    return (time.perf_counter() - t0) * 1000.0 / n_images

def run_cnn(split_type, fold, train_df, val_df, test_df):
    tf.keras.backend.clear_session()
    model, hist, th, train_time = train_cnn(train_df, val_df)

    test_ds = make_ds(test_df, training=False, augment=False)
    y_true  = test_df['binary_label'].values
    y_prob  = model.predict(test_ds, verbose=0).ravel()

    fold_tag = 'NA' if pd.isna(fold) else int(fold)
    exp_dir  = make_experiment_dir(split_type, fold)      # experiments/<timestamp>_.../
    ckpt = exp_dir/'model.keras'
    model.save(ckpt)
    sizes = dict(n_train=len(train_df), n_val=len(val_df), n_test=len(test_df))
    save_training_log(exp_dir, hist, th)                  # training_log.csv (andamento fit)
    save_experiment_config(exp_dir, split_type, fold, sizes)  # config.json (iperparametri)

    row = dict(dataset_config=CONFIG_NAME, split_type=split_type, fold=fold,
               model='baseline_cnn', experiment=exp_dir.name)
    row.update(compute_metrics(y_true, y_prob))
    row.update(
        training_time_sec=round(train_time, 2),
        avg_epoch_time_sec=round(float(np.mean(th.times)), 2) if th.times else np.nan,
        inference_time_ms_per_image=round(measure_inference_ms(model, test_ds, len(test_df)), 3),
        n_params=int(model.count_params()),
        model_size_mb=round(ckpt.stat().st_size / 1e6, 2),
        n_train=len(train_df), n_val=len(val_df), n_test=len(test_df),
        epochs_run=len(th.times), notes='',
    )
    pd.DataFrame([row]).to_csv(exp_dir/'metrics.csv', index=False)  # metriche del singolo run
    print(f'[CNN] {split_type} fold={fold_tag}: '
          f"AUROC={row['auroc'] if pd.isna(row['auroc']) else round(row['auroc'],3)} "
          f"acc={row['accuracy']:.3f} rec_mal={row['recall_malignant']:.3f} "
          f"({row['epochs_run']} epoche, {row['training_time_sec']:.0f}s)")
    del model; gc.collect()
    return row

## Sezione 6 — Enumerazione dei protocolli di valutazione

Un'unica lista di run `(split_type, fold, train, val, test)` costruita **solo** dai CSV di Persona 1:
image-wise (1 split), patient-wise (1 split), k-fold patient-wise (5 fold). Nessuna rigenerazione.

In [ ]:
def enumerate_runs():
    runs = []

    iw = add_full_path(pd.read_csv(CONFIG_DIR/'image_wise_split.csv'))
    runs.append(('image_wise', np.nan,
                 iw[iw.split=='train'], iw[iw.split=='val'], iw[iw.split=='test']))

    pw = add_full_path(pd.read_csv(CONFIG_DIR/'patient_wise_split.csv'))
    runs.append(('patient_wise', np.nan,
                 pw[pw.split=='train'], pw[pw.split=='val'], pw[pw.split=='test']))

    kf = add_full_path(pd.read_csv(CONFIG_DIR/'patient_wise_folds.csv'))
    for fo in sorted(kf['fold'].unique()):
        f = kf[kf.fold == fo]
        runs.append(('kfold_patient_wise', int(fo),
                     f[f.split=='train'], f[f.split=='val'], f[f.split=='test']))
    return runs

RUNS = enumerate_runs()
print('Protocolli di valutazione:')
for st, fo, tr, va, te in RUNS:
    ft = 'NA' if pd.isna(fo) else int(fo)
    print(f'  {st:20s} fold={ft:>2}  train={len(tr):4d}  val={len(va):3d}  test={len(te):3d}  '
          f"(test pazienti: {te['patient_id'].nunique()})")

## Sezione 7 — Baseline CNN su tutti i protocolli

Attenzione ai tempi: su GPU Colab, ~5-8 min per fold; l'intero blocco (7 run) puo richiedere
30-60 min. Usa `QUICK_TEST=True` nella Sezione 0 per una prova di correttezza in pochi minuti.

**Output per run.** Ogni fold produce una cartella `experiments/<timestamp>_<config>_<split>_baseline_cnn_scratch_fold<N>/` con `config.json`, `model.keras`, `training_log.csv` e `metrics.csv` (schema allineato a Persona 2). Le tabelle aggregate di confronto restano in `results/tables/` (Sez. 8).

In [ ]:
cnn_results = [run_cnn(st, fo, tr, va, te) for (st, fo, tr, va, te) in RUNS]
print('\nCNN completata:', len(cnn_results), 'run.')

## Sezione 8 — Tabella finale e sintesi k-fold (per D1/D2)

`results/tables/cnn_baseline_metrics.csv` con lo schema richiesto (colonne extra tn/fp/fn/tp e
dimensioni utili in coda). In piu, la sintesi media +/- std sulle 5 fold della CNN:
e questo il numero da riportare per D2, non il singolo split.

In [ ]:
REQUIRED = ['dataset_config','split_type','fold','model','accuracy','precision','recall',
            'f1','auroc','recall_malignant','training_time_sec','avg_epoch_time_sec',
            'inference_time_ms_per_image','n_params','model_size_mb']
EXTRA = ['tn','fp','fn','tp','n_train','n_val','n_test','epochs_run','experiment','notes']

df = pd.DataFrame(cnn_results)
df = df.reindex(columns=REQUIRED + EXTRA)
out_csv = TABLES_DIR/'cnn_baseline_metrics.csv'
df.to_csv(out_csv, index=False)
print('Salvato:', out_csv, '|', df.shape)

# ---- Sintesi k-fold: media +/- std per modello (D2) ------------------------
kf = df[df.split_type == 'kfold_patient_wise']
agg_cols = ['accuracy','precision','recall','f1','auroc','recall_malignant',
            'training_time_sec','inference_time_ms_per_image']
summary = (kf.groupby('model')[agg_cols]
             .agg(['mean','std']).round(4))
summary.columns = [f'{m}_{s}' for m, s in summary.columns]
summary = summary.reset_index()
summary.insert(0, 'dataset_config', CONFIG_NAME)
summary.insert(2, 'split_type', 'kfold_patient_wise')
summary.insert(3, 'k', kf['fold'].nunique())
summary_csv = TABLES_DIR/'cnn_baseline_kfold_summary.csv'
summary.to_csv(summary_csv, index=False)
print('Salvato:', summary_csv)
summary[['model','auroc_mean','auroc_std','recall_malignant_mean','recall_malignant_std']]

In [ ]:
# Confusion matrix aggregata sulle 5 fold (solo CNN) per lettura D2
cnn_kf = df[(df.split_type=='kfold_patient_wise') & (df.model=='baseline_cnn')]
tn, fp, fn, tp = [int(cnn_kf[c].sum()) for c in ('tn','fp','fn','tp')]
cm = np.array([[tn, fp], [fn, tp]])

fig, ax = plt.subplots(figsize=(4.2, 3.8))
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks([0,1], ['benign','malignant']); ax.set_yticks([0,1], ['benign','malignant'])
ax.set_xlabel('Predetto'); ax.set_ylabel('Reale')
ax.set_title(f'CNN k-fold (somma 5 fold) — {CONFIG_NAME}')
for i in range(2):
    for j in range(2):
        ax.text(j, i, cm[i, j], ha='center', va='center',
                color='white' if cm[i, j] > cm.max()/2 else 'black', fontsize=13)
fig.tight_layout()
fig.savefig(FIGURES_DIR/'cnn_kfold_confusion_matrix.png', dpi=150)
plt.show()
print('Recall maligni aggregata (TP/(TP+FN)):', round(tp/(tp+fn), 3))

## Sezione 9 — Lettura dei risultati per D2

- **image-wise**: stima potenzialmente **ottimistica** (patch dello stesso paziente possono cadere
  in train e test) — utile solo come limite superiore, non come misura di generalizzazione.
- **patient-wise (singolo split)**: piu rigoroso, ma con ~5 pazienti in test la varianza e alta:
  un singolo numero non e affidabile.
- **k-fold patient-wise (media +/- std)**: e il riferimento per D2. La deviazione standard tra fold
  quantifica l'incertezza dovuta al basso numero di pazienti e va **sempre** riportata accanto alla media.
- Per la classe maligna privilegiare **recall_malignant** (sensibilita): nel dominio clinico un falso
  negativo pesa piu di un falso positivo.

**Consegna a Persona 2.** `cnn_baseline_metrics.csv` e la sintesi k-fold sono i termini di paragone
per il transfer learning: Persona 2 deve valutare **sulle stesse fold** (D1 = CNN da zero vs transfer,
a parita di split/efficienza).